# 03 — Creator Segmentation

**Purpose:** Segment creators into actionable cohorts for partnerships teams.

This notebook answers:
- Which creator cohorts show the strongest engagement consistency?
- Which cohorts are strongest for awareness vs engagement objectives?
- How does performance vary by category, tier, and geography?

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from ingest_real_data import load_scored_channels
from utils import set_plot_style, COLORS, TIER_ORDER, TIER_COLORS, format_number

set_plot_style()

In [ ]:
df = load_scored_channels()
print(f'Loaded {len(df)} scored channels')
print(f'Channels with complete scores: {df["creator_fit_score"].notna().sum()}')

## 1. Tier-Level Segmentation

In [ ]:
tier_summary = df.groupby('follower_tier').agg(
    channels=('channel_name', 'count'),
    avg_subs=('subscribers', lambda x: f"{x.mean()/1e6:.1f}M"),
    avg_views_per_sub=('engagement_proxy', 'mean'),
    avg_momentum=('momentum_score', 'mean'),
    avg_consistency=('consistency_score', 'mean'),
    avg_fit_score=('creator_fit_score', 'mean'),
).reindex(TIER_ORDER).round(1)

tier_summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

tier_grouped = df.groupby('follower_tier')

for ax, col, title in zip(axes,
    ['engagement_proxy', 'momentum_score', 'consistency_score'],
    ['Engagement Proxy', 'Momentum Score', 'Consistency Score']):
    data = [df[df['follower_tier'] == t][col].dropna() for t in TIER_ORDER if t in df['follower_tier'].values]
    labels = [t for t in TIER_ORDER if t in df['follower_tier'].values]
    bp = ax.boxplot(data, labels=labels, patch_artist=True)
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(TIER_COLORS[i] if i < len(TIER_COLORS) else COLORS['neutral'])
        patch.set_alpha(0.6)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Key Metrics by Follower Tier', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../dashboard/screenshots/tier_segmentation.png')
plt.show()

## 2. Category-Level Segmentation

In [ ]:
cat_summary = df[df['category'].notna()].groupby('category').agg(
    channels=('channel_name', 'count'),
    avg_subs=('subscribers', 'mean'),
    avg_engagement=('engagement_proxy', 'mean'),
    avg_momentum=('momentum_score', 'mean'),
    avg_consistency=('consistency_score', 'mean'),
    avg_fit_score=('creator_fit_score', 'mean'),
).sort_values('avg_fit_score', ascending=False).round(1)

cat_summary[cat_summary['channels'] >= 5]

In [ ]:
# Top categories by engagement vs momentum
cat_plot = cat_summary[cat_summary['channels'] >= 10].copy()

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(
    cat_plot['avg_engagement'], cat_plot['avg_momentum'],
    s=cat_plot['channels'] * 8, alpha=0.7, color=COLORS['primary'],
    edgecolors='white', linewidth=1
)

for idx, row in cat_plot.iterrows():
    ax.annotate(idx, (row['avg_engagement'], row['avg_momentum']),
                fontsize=9, ha='center', va='bottom')

ax.set_xlabel('Avg Engagement Proxy (Views/Sub)')
ax.set_ylabel('Avg Momentum Score')
ax.set_title('Category Segmentation: Engagement vs Momentum\n(bubble size = number of channels)')
plt.tight_layout()
plt.savefig('../dashboard/screenshots/category_segmentation.png')
plt.show()

## 3. Geographic Distribution

In [ ]:
country_summary = df[df['country'].notna()].groupby('country').agg(
    channels=('channel_name', 'count'),
    avg_subs=('subscribers', 'mean'),
    avg_engagement=('engagement_proxy', 'mean'),
    total_views=('total_views', 'sum'),
).sort_values('channels', ascending=False).round(1)

top_countries = country_summary.head(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_countries.index[::-1], top_countries['channels'][::-1],
        color=COLORS['primary'], alpha=0.7)
ax.set_xlabel('Number of Top Channels')
ax.set_title('Top 15 Countries by Channel Count')
plt.tight_layout()
plt.savefig('../dashboard/screenshots/geographic_distribution.png')
plt.show()

## 4. Awareness vs Engagement Segmentation

The key partnerships visual: which channels are strong on reach, engagement, or both?

In [ ]:
scored = df[df['awareness_score'].notna() & df['engagement_suitability_score'].notna()].copy()

fig, ax = plt.subplots(figsize=(11, 8))

tier_color_map = dict(zip(TIER_ORDER, TIER_COLORS))
colors = scored['follower_tier'].map(tier_color_map).fillna(COLORS['neutral'])

ax.scatter(
    scored['awareness_score'], scored['engagement_suitability_score'],
    c=colors, alpha=0.6, s=40, edgecolors='white', linewidth=0.5
)

aw_med = scored['awareness_score'].median()
en_med = scored['engagement_suitability_score'].median()
ax.axvline(aw_med, color=COLORS['neutral'], linestyle='--', alpha=0.5)
ax.axhline(en_med, color=COLORS['neutral'], linestyle='--', alpha=0.5)

ax.text(88, 78, 'STAR CREATORS\nHigh Reach + Engagement', ha='center', fontsize=9,
        color=COLORS['success'], fontweight='bold')
ax.text(12, 78, 'ENGAGEMENT\nSPECIALISTS', ha='center', fontsize=9,
        color=COLORS['secondary'], fontweight='bold')
ax.text(88, 12, 'AWARENESS\nDRIVERS', ha='center', fontsize=9,
        color=COLORS['primary'], fontweight='bold')
ax.text(12, 12, 'NICHE /\nEMERGING', ha='center', fontsize=9,
        color=COLORS['neutral'], fontweight='bold')

ax.set_xlabel('Awareness Score')
ax.set_ylabel('Engagement Suitability Score')
ax.set_title('Creator Segmentation: Awareness vs Engagement')

handles = [mpatches.Patch(color=c, label=t) for t, c in tier_color_map.items()]
ax.legend(handles=handles, title='Follower Tier', loc='lower right')

plt.tight_layout()
plt.savefig('../dashboard/screenshots/awareness_vs_engagement.png')
plt.show()

## 5. Segment Counts

In [ ]:
def assign_segment(row):
    if pd.isna(row['awareness_score']) or pd.isna(row['engagement_suitability_score']):
        return 'Insufficient Data'
    aw_med = scored['awareness_score'].median()
    en_med = scored['engagement_suitability_score'].median()
    if row['awareness_score'] >= aw_med and row['engagement_suitability_score'] >= en_med:
        return 'Star Creator'
    elif row['awareness_score'] >= aw_med:
        return 'Awareness Driver'
    elif row['engagement_suitability_score'] >= en_med:
        return 'Engagement Specialist'
    else:
        return 'Niche / Emerging'

scored['segment'] = scored.apply(assign_segment, axis=1)

segment_summary = scored.groupby('segment').agg(
    channels=('channel_name', 'count'),
    avg_subs=('subscribers', lambda x: f"{x.mean()/1e6:.1f}M"),
    avg_fit_score=('creator_fit_score', 'mean'),
    avg_awareness=('awareness_score', 'mean'),
    avg_engagement=('engagement_suitability_score', 'mean'),
).round(1)

segment_summary

---

## Summary for Partnerships Teams

1. **Star Creators** are rare — channels scoring high on both awareness and engagement are the most valuable and hardest to find
2. **Category matters more than size** for engagement-focused campaigns — some mid-size categories significantly outperform larger ones
3. **Geographic concentration** is real — India and the US dominate the top channel landscape, which matters for market-specific campaigns
4. **Momentum is not correlated with size** — some smaller channels are growing faster than mega channels, making them strong candidates for early partnerships